# Forecast experiment (Stage 10)

This notebook walks through the 30-minute glucose forecast step by step: the data, the features, the two baselines, the gradient boosting model and how the errors are measured. The same steps run in `python -m glucobalance.forecast`, which writes `docs/forecast/report.md`; this notebook is for looking at them.

Open it with `uv run --with jupyterlab jupyter lab notebooks/forecast_experiment.ipynb`. The charts need Plotly in the notebook kernel (`--with plotly`).

All data here is synthetic. Nothing comes from a real person.

## 1. A synthetic patient

One reading every 5 minutes, with meals, boluses and a little chaos.

In [ ]:
from glucobalance.forecast.history import synthetic_history

history = synthetic_history(days=3, seed=1)
values = [v for _, v in history.readings]
print(len(history.readings), "readings,", len(history.carbs), "carb entries,")
print(len(history.boluses), "boluses")
print("lowest", min(values), "highest", max(values), "mean", round(sum(values) / len(values)))

In [ ]:
import plotly.graph_objects as go

figure = go.Figure(go.Scatter(x=[t for t, _ in history.readings], y=values, name="CGM"))
for at, _grams in history.carbs:
    figure.add_vline(x=at, line_color="orange", opacity=0.3)
figure.add_hrect(y0=70, y1=180, fillcolor="teal", opacity=0.08, line_width=0)
figure.update_layout(title="Three synthetic days (orange lines are meals)", yaxis_title="mg/dL")
figure

## 2. Features

Each reading becomes one row of numbers, using only what was known at that moment. The target is the change 30 minutes later.

In [ ]:
from glucobalance.forecast.features import FEATURE_NAMES, build_samples

samples = build_samples(history)
example = samples[200]
print(example.at, example.glucose, "mg/dL, change in 30 min:", example.target)
for name, value in zip(FEATURE_NAMES, example.features, strict=True):
    print(f"  {name:<10} {value:8.2f}")

## 3. Train and test

Eight patients to train on (first 70% of their days), the rest of their days and three new patients to test on. Splitting by time, not at random, keeps the test honest.

In [ ]:
from glucobalance.forecast.experiment import run_experiment

train = [synthetic_history(30, seed) for seed in range(1, 9)]
unseen = [synthetic_history(30, seed) for seed in range(1001, 1004)]
result = run_experiment(train, unseen, source="synthetic", days=30)
for title, scores in (("later days", result.later), ("unseen patients", result.unseen)):
    print(title)
    for s in scores:
        print(f"  {s.model:<18} RMSE {s.rmse:5.1f}  MAE {s.mae:5.1f}")

## 4. The Clarke error grid

Each dot is one forecast. Zone A is within 20% of the real value; zones C to E are the dangerous ones.

In [ ]:
from glucobalance.forecast.evaluation import ClarkeZone, clarke_zone

test = [s for h in unseen for s in build_samples(h)]
actual = [s.glucose + (s.target or 0) for s in test]
predicted = result.model.predict([s.features for s in test])
zones = [clarke_zone(a, p) for a, p in zip(actual, predicted, strict=True)]
for zone in ClarkeZone:
    print(zone.value, f"{zones.count(zone) / len(zones):.1%}")

dots = go.Scattergl(x=actual, y=predicted, mode="markers", marker={"size": 3, "opacity": 0.3})
grid = go.Figure(dots)
grid.add_shape(type="line", x0=0, y0=0, x1=400, y1=400, line={"dash": "dot"})
grid.update_layout(
    title="Gradient boosting, unseen patients",
    xaxis_title="actual mg/dL in 30 min",
    yaxis_title="predicted mg/dL",
    width=600,
    height=600,
)
grid

## 5. The warning line

How many lows are caught, and how many warnings are right, as the warning line moves. The app warns below 80 mg/dL.

In [ ]:
from glucobalance.forecast.evaluation import low_detection

current = [float(s.glucose) for s in test]
for line in (70, 75, 80, 85, 90):
    found = low_detection(current, actual, predicted, warn_below=line)
    print(f"warn below {line}: caught {found.sensitivity:.0%}, right {found.precision:.0%}")

## 6. What this does not show

- Real CGM data is noisier and has more unlogged events, so real errors are larger (published 30-minute RMSEs are roughly 18 to 30 mg/dL).
- `uv run --group sim python -m glucobalance.forecast --source simglucose` repeats the experiment on the UVA/Padova simulator for a harder test.
- See `docs/forecast/model-card.md` for the intended use and limits.